# Aortic inertance (RLC) sweep — batched recalibration (Reviewer 1, R1-C3)

<details>
<summary>Does the EFC tuning approach need changes for RLC-based lumped-parameter models?</summary>

Reviewer 1 (R1-C3) notes the model is R+C only, with no inductance (inertance)
elements for blood inertia. This notebook adds an inductor at the **aortic (LV)
outflow** — the valve `Hl_As`, flipped from `diode` to `diode_inertial` in
`cvModel_linear_inertial.json`, which promotes the aortic flow `Q_Hl_As` to an
integrated state with `dQ/dt = (P_Hl - P_As - Q·R)/L` — then **sweeps the
inductance `L` across a geometric range in one vmapped batch** and lets the twin
calibration controllers **recalibrate to the same fixed physiological twin
targets at every L level**. The unmodified `diode` model
(`cvModel_linear.json`) is run once as the L→0 pure-RC anchor.

The argument to the reviewer: the tuning approach is **unchanged** — inertance
is just another integrated forward-model state; the controllers act on
observed-signal errors and recalibrate transparently, so the calibration
residual stays flat/low across the whole L range. Two-phase run/plot; the
population `.h5` is the artifact and the figure/table emit into the paper
revision tree.

</details>

In [ ]:
# region -> runConfig — the single run-configuration surface (device/precision applied before JAX)
# The ONE place run configuration lives (project rule: repo CLAUDE.md). Defined first so the
# device/precision block applies before JAX initialises in the Imports cell.
runConfig = {
    # --- file references ---
    "model":    "cvModel_linear_inertial.json",  # aortic valve Hl_As = diode_inertial (adds Q_Hl_As state)
    "scenario": "sepsis_linear.json",            # twin targets + calibration + convergence obs list
    "mode":     "calibration",                   # staged calibration to the fixed twin targets

    # --- pipeline phases (run + plot separable) ---
    "run":  False,   # phase 1 — run the L sweep + RC anchor, save the artifacts.
                     #   Artifacts already exist under output.path; flip True to re-run the
                     #   64-lane sweep (~13 min CPU) + RC anchor from scratch.
    "plot": True,    # phase 2 — load + analyse + figure + table

    # --- device / precision (applied in Imports cell, before `import jax`) ---
    "device": {
        "useGpu":    False,       # GPU is OFF by default everywhere (repo HARD RULE)
        "precision": "float64",   # "float64" or "float32"
    },

    # --- inductance sweep (NEW knob) — the L axis of the batch ---
    "inductance": {
        "state":    "L_Hl_As",   # inductor state to sweep (aortic outflow)
        "nrModels": 64,          # lanes = distinct L values in the batch
        "min":      0.0005,      # near-RC limit (mmHg·s²/mL)
        "max":      0.1,         # ~10x exaggerated inertance
        "spacing":  "geometric", # "geometric" (log-spaced) or "linear"
    },

    # --- RC anchor: the unmodified diode model as the L->0 pure-RC reference ---
    "rcAnchor": {
        "enabled": True,
        "model":   "cvModel_linear.json",   # same physiology, Hl_As = diode (no inertance)
    },

    # --- batched solve ---
    "solver":    {"type": "euler"},  # "euler" or "rk4"; keep euler (dt == step) on the integrated stack
    "chunkSize": 64,                 # samples per vmap (VRAM bound)
    "calibration": {},               # per-key merge over scenario calibration ({} = as-is)

    # --- analysis ---
    "analysis": {
        "atm":                 760.0,     # atmospheric offset for absolute-pressure signals
        "divergenceLimit":     50000.0,   # |value| >= this in any obs/param -> BAD lane
        "errorTarget":         0.5,       # SUCCESS if max |rel err| (%) <= this
        "nParamsToPlot":       4,         # most L-sensitive params HIGHLIGHTED vs L (the rest drawn faint)
        "showFliers":          True,      # draw the lanes outside the whiskers (the error tail)
        "showErrorTargetLine": False,     # errorTarget line on the residual panel; off because no lane
                                          #   meets 0.5% (the RC anchor is 1.79%) — the RC dashed lines
                                          #   are the meaningful reference for this sweep
    },

    "progressEvery": 0,     # live convergence line every N sim seconds (0 = off for the sweep)
    "saveRaw":       True,  # stream every lane's trajectory (Phase 2 reads run-end from `raw`)

    # --- output (sweep + RC anchor artifacts) ---
    "output": {
        "save":       True,
        "path":       "notebookData/inductance",
        "name":       "inductance_sweep.h5",       # sweep artifact
        "rcName":     "inductance_rc_anchor.h5",   # RC-anchor artifact
        "logProgress": True,
    },

    # --- paper artifact routing (guarded by paper.emit) ---
    "paper": {
        "emit":         True,
        "generatedDir": "EFC_Paper/responseLetter/generated",
        "imagesDir":    "EFC_Paper/responseLetter/generated",
        "tableName":    "inductanceSweep.tex",
        "tableLabel":   "Table~R1.",   # manual label of the emitted non-float table (letter style)
        "boldPct":      10.0,          # rows whose range exceeds this % of the paper value are bolded
        "figName":      "inductanceSweep.png",
        "figBlockName": "inductanceSweepFig.tex",   # figure + caption block \input by the letter
        "figLabel":     "Fig.~R2.",    # manual label of the emitted figure block (letter style)
    },

    "postProcessing": None,
    "plots":          [],
    "plotOpts":       {"targetPoints": 1500},   # strided-decimation target for the full-raw convergence plots
    "printStatus":    True,
    "printEveryPct":  100,

    # --- integration numerics (override scenario shared.integration) ---
    "runTime": 10,       # simulated seconds per internal run
    "dt":      0.0005,   # integrator step (== cycle step on the integrated stack)
    "dtDense": 1.0,      # save grid: 1 Hz run-end sampling is enough for the residual read
}
# endregion

## Imports

In [ ]:
# region -> imports + device/precision (must precede `import jax`)
# ---- repo-root bootstrap: run from any cwd (make `library` importable + resolve the relative
# ---- notebookData/ + config/ paths). Walks up to the dir containing library/. ----
import os, sys
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "library")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)

# ---- device / precision (from runConfig, MUST run before JAX initialises) ----
useGpu    = runConfig["device"]["useGpu"]
precision = runConfig["device"]["precision"]

if useGpu:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)
    os.environ["JAX_PLATFORMS"] = "cuda"
    os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
    os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]   = "platform"
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
    os.environ["JAX_PLATFORMS"] = "cpu"

import jax
jax.config.update("jax_enable_x64", precision == "float64")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, time

import library.run.runner as runner            # buildSimulationParams
import library.run.runnerBatchSI as runnerBatchSI   # batched (vmapped) SI calibration
import library.viz.plots as libPlots           # plotInductanceSweep
import library.utils as utils
from library.hdf5 import schema_pop                       # population artifact
from library.hdf5.raw_stream import RawTraceStreamWriter  # async raw streaming writer
import library.postproc.reporting as reporting            # shared scope / rejection report

np.set_printoptions(suppress=True)
print("devices:", jax.devices(), "| x64:", jax.config.jax_enable_x64)
# endregion

## Assemble `simulationParams` + load the twin/observation config

<details>
<summary>Load the scenario twin + convergence observation list; the sweep axis is L_Hl_As only.</summary>

Unlike the population convergence driver, we do **not** LHS the 16 calibrated
parameters — they start from the model base and the controllers drive them to
the fixed twin targets. The only swept column is the inductor state
`inductance.state`; every lane shares the same twin targets.

</details>

In [ ]:
# region -> assemble simulationParams + load the twin/observation config (sweep axis = L only)
scenario = utils.loadScenario(runConfig["scenario"])
simulationParams = runner.buildSimulationParams(runConfig, scenario)

conv          = scenario["convergence"]
calConf       = simulationParams["simulationConf"]["calibration"]
twin          = scenario["shared"]["twin"]["twinTargets"]
volDist       = scenario["shared"]["twin"]["volumeDistribution"]
observations  = conv["observations"]
calibParams   = calConf["adaptive"]["parameters"]        # the 16 calibrated params (run-end read)
ind           = runConfig["inductance"]
Lname         = ind["state"]
outDir        = runConfig["output"]["path"]
sweepPath     = os.path.join(outDir, runConfig["output"]["name"])
rcPath        = os.path.join(outDir, runConfig["output"]["rcName"])

print(f"sweep state = {Lname} | lanes = {ind['nrModels']} | observations = {len(observations)} "
      f"| calibrated params = {len(calibParams)}")
print(f"sweep -> {sweepPath}\nrc anchor -> {rcPath}")
# endregion

## Twin-target array + atmospheric offsets

<details>
<summary>Per-observation twin target (gauge) + atmospheric offset — the same fixed targets for every lane.</summary>

The capillary means are derived as a pressure DROP from the upstream arterial
target (matching the convergence driver), not as absolute means.

</details>

In [ ]:
# region -> twin-target array + atmospheric offsets per observation
ATM = runConfig["analysis"]["atm"]

def obsOffset(name):
    """Atmospheric offset baked into absolute-pressure signals (gauge = raw - offset)."""
    return utils.obsOffset(name, ATM)

def obsTarget(name):
    """Twin target for an observation (gauge units), or None if untargeted."""
    TBV = twin["TotalBloodVolume"]
    direct = {
        "avg_P_Vs": twin["CVP"],
        "avg_P_Cs": twin["Sys_P_As"] - twin["amp_P_As"] - twin["avg_P_Cs"],
        "avg_P_Cp": twin["Dia_P_Ap"] - twin["avg_P_Cp"],
        "keep_max_P_As": twin["Sys_P_As"], "keep_max_P_Ap": twin["Sys_P_Ap"],
        "keep_min_P_Ap": twin["Dia_P_Ap"], "amp_P_As": twin["amp_P_As"],
        "keep_SV_Hl": twin["CO"] / twin["HR"], "Cyc_HC": 60.0 / twin["HR"],
        "V_Vs": volDist["Vs"] * TBV,
    }
    if name in direct:
        return direct[name]
    if name.startswith("avg_V_"):
        return volDist[name[len("avg_V_"):]] * TBV
    return None

targetArr = np.array([obsTarget(o) if obsTarget(o) is not None else np.nan for o in observations])
offsetArr = np.array([obsOffset(o) for o in observations])
pd.DataFrame({"observation": observations, "target": targetArr, "offset": offsetArr})
# endregion

## Inductance sweep axis (geometric L)

<details>
<summary>Build the (nrModels, 1) column of L values injected into the Y0 L_Hl_As state per lane.</summary>

`L_Hl_As` is a `NoController` parameter-variation state, so an injected value is
held fixed for the whole run while the twin controllers recalibrate around it.

</details>

In [ ]:
# region -> inductance sweep axis (geometric L column injected into the L_Hl_As state)
if runConfig["run"]:
    if ind.get("spacing", "geometric") == "geometric":
        Lvals = np.geomspace(ind["min"], ind["max"], ind["nrModels"])
    else:
        Lvals = np.linspace(ind["min"], ind["max"], ind["nrModels"])
    sampled_params = Lvals.reshape(-1, 1)                 # (N, 1); column maps to [Lname]
    param_names = [Lname]
    print(f"L sweep: {ind['nrModels']} values in [{ind['min']}, {ind['max']}] "
          f"({ind.get('spacing','geometric')})")
    pd.DataFrame({Lname: Lvals}).describe().T
# endregion

## Run — batched L sweep + RC anchor (single vmapped solve each)

<details>
<summary>One batchedCalibration for the 64-lane L sweep (inertial model) + a 1-lane RC anchor (diode model).</summary>

Both recalibrate to the SAME fixed twin targets. The RC anchor injects nothing
(`param_names=[]`), so it calibrates the pure-RC model from base. Each artifact
is a standalone population `.h5` (model structure + raw tensor + final states).

</details>

In [ ]:
# region -> run the batched L sweep + RC anchor, save both population artifacts
def _runPop(modelName, sampled, names, path, tag):
    """Run one vmapped calibration population for `modelName` and save it via schema_pop.
    sampled: (N, P) injection matrix; names: P state names (== columns). Returns the batch dict."""
    cfg = {**runConfig, "model": modelName}
    sp  = runner.buildSimulationParams(cfg, scenario)
    prep   = runnerBatchSI.prepare(sp)
    layout = runnerBatchSI.rawLayout(sp, prepared=prep)
    stateNames = layout["stateNames"]
    rawDtype   = "float64" if jax.config.jax_enable_x64 else "float32"
    N = sampled.shape[0]
    traceNames = list(dict.fromkeys(list(observations) + list(names)))
    saveRaw = runConfig.get("saveRaw", False) and runConfig["output"]["save"]

    if runConfig["output"]["save"]:
        os.makedirs(runConfig["output"]["path"], exist_ok=True)
        schema_pop.init_population(
            path, param_names=names, state_names=stateNames,
            observation_names=observations, sampled_params=sampled,
            model_structure=utils.modelStructureJSON(prep["modelStructure"]),
            problem={"names": names, "bounds": [], "num_vars": len(names)},
            conf=cfg, meta={"twinTargets": twin, "tag": tag})

    def rawWriterFactory(signalNames, totalPoints, time_, nDense):
        return RawTraceStreamWriter(path, N=N, signalNames=signalNames,
                                    totalPoints=totalPoints, nDense=nDense, time=time_, dtype=rawDtype)

    t0 = time.time()
    batch = runnerBatchSI.batchedCalibration(
        sp, sampled, names, observations,
        chunkSize=runConfig.get("chunkSize", 64), printStatus=runConfig.get("printStatus", True),
        printEveryPct=runConfig.get("printEveryPct"), traceNames=traceNames,
        rawWriterFactory=(rawWriterFactory if saveRaw else None), prepared=prep)
    wall = time.time() - t0
    print(f"[{tag}] {modelName}: N={N} solved in {wall:.1f}s (solver={sp['solver']['type']})")

    if runConfig["output"]["save"]:
        schema_pop.write_final_states(path, np.asarray(batch["finalStates"]),
                                      [str(i) for i in range(N)])
        schema_pop.write_timings(path, [wall], meta={
            "device": "gpu" if useGpu else "cpu", "precision": precision,
            "solver": sp["solver"]["type"], "dt": sp["dt"], "runTime": sp["runTime"],
            "nrModels": N, "stack": "SI", "total_wall": wall, "tag": tag})
        schema_pop.write_progress(path, batch.get("progress"), meta={
            "model": modelName, "scenario": runConfig["scenario"], "mode": runConfig["mode"],
            "solver": sp["solver"]["type"], "nrModels": N, "stack": "SI"})
    return batch

if runConfig["run"]:
    _ = _runPop(runConfig["model"], sampled_params, param_names, sweepPath, "sweep")
    if runConfig["rcAnchor"]["enabled"]:
        _ = _runPop(runConfig["rcAnchor"]["model"], np.zeros((1, 0)), [], rcPath, "rc")
    print("run phase complete.")
# endregion

# Phase 2 — Load & analyse (from the saved files)

<details>
<summary>Reconstruct the canonical population state (modelStructure, obsMatrix, finiteRow, timings) from each artifact.</summary>

Same load contract as the convergence/batch driver so the standard population
plots below apply unchanged: obsMatrix[l, j] = last raw timepoint of observation
j minus its atmospheric offset (== the serial steady-state read). The swept
"parameter" here is the aortic inductance `L_Hl_As` (one lane per L); the 16
calibrated params are read at run-end for the convergence panels. The L per lane
is recomputed from the sweep axis; the RC anchor is a single-lane reference.

</details>

In [ ]:
# region -> load the canonical population state (obsMatrix / finiteRow / timings) from each artifact
def _loadPop(path):
    """Return (obsMatrix (N,nObs), lastRow (N,C), sigIdx, sig, modelStructure) from a saved file."""
    import h5py
    with h5py.File(path, "r") as f:
        sig    = list(f["raw_signal_names"].asstr()[:])
        ms     = json.loads(f["model_structure"].asstr()[()])
        src    = "raw_coarse" if "raw_coarse" in f else "raw"
        lastRow = np.asarray(f[src][:, -1, :])            # (N, C) run-end values
    sigIdx = {n: i for i, n in enumerate(sig)}
    obsM = np.full((lastRow.shape[0], len(observations)), np.nan)
    for j, o in enumerate(observations):
        if o in sigIdx:
            obsM[:, j] = lastRow[:, sigIdx[o]] - obsOffset(o)
    return obsM, lastRow, sigIdx, sig, ms

if runConfig["plot"]:
    outPath = sweepPath                                   # the sweep is the primary population
    pop = {"nrModels": ind["nrModels"], "errorTarget": runConfig["analysis"]["errorTarget"]}

    # L per lane (deterministic from the sweep axis)
    Lvals = (np.geomspace(ind["min"], ind["max"], ind["nrModels"])
             if ind.get("spacing", "geometric") == "geometric"
             else np.linspace(ind["min"], ind["max"], ind["nrModels"]))

    obsMatrix, lastRow, sigIdx, sig, modelStructure = _loadPop(outPath)
    Nrun   = lastRow.shape[0]
    divLim = runConfig["analysis"]["divergenceLimit"]

    # calibrated params (the 16) read at run-end -> checked for out-of-scope + used by the plots
    plotParams  = [p for p in calibParams if p in sigIdx]
    paramMatrix = (np.column_stack([lastRow[:, sigIdx[p]] for p in plotParams])
                   if plotParams else np.zeros((Nrun, 0)))
    finiteRow = schema_pop.good_run_mask(obsMatrix, divLim, param_matrix=paramMatrix)
    obsMatrix[~finiteRow] = np.nan
    goodSweep = finiteRow                                  # sweep-specific alias

    traceT, traces, saveRaw = None, None, True            # convergence plots read raw/raw_coarse below
    runWall, timingMeta = schema_pop.read_timings(outPath)

    # RC anchor (single lane, pure-RC reference)
    rcObs, rcParams = None, None
    if runConfig["rcAnchor"]["enabled"] and os.path.exists(rcPath):
        rcObs, rcLast, rcSigIdx, _, _ = _loadPop(rcPath)
        # the unmodified model's calibrated parameters = the paper values (Table 5)
        rcParams = {p: float(rcLast[0, rcSigIdx[p]]) for p in plotParams if p in rcSigIdx}

    print(f"loaded {Nrun} lanes from {outPath} | converged {finiteRow.sum()}/{Nrun} "
          f"| timings: {'yes' if timingMeta else 'none'}")
# endregion

## Error summary

<details>
<summary>Drop faulty lanes; per-observation relative/absolute error across the converged sweep.</summary>

</details>

In [ ]:
# region -> error summary — drop faulty lanes, per-observation relative/absolute error
if runConfig["plot"]:
    targeted = ~np.isnan(targetArr)                     # observations that carry a twin target
    good = schema_pop.good_run_mask(obsMatrix, divLim, param_matrix=paramMatrix)
    print(f"{good.sum()}/{len(good)} valid lanes ({(~good).sum()} faulty/diverged dropped)")

    obsGood = obsMatrix[good][:, targeted]
    tgt = targetArr[targeted]
    errRel = (obsGood - tgt) / tgt * 100.0
    errAbs = obsGood - tgt
    obsTargeted = [o for o, t in zip(observations, targeted) if t]

    success = np.max(np.abs(errRel), axis=1) <= pop["errorTarget"]
    print(f"within max|rel err| <= {pop['errorTarget']}%: {success.sum()}/{len(success)} lanes")

    summary = pd.DataFrame({
        "observation": obsTargeted,
        "target": tgt,
        "mean_rel_err_%": np.nanmean(errRel, axis=0),
        "std_rel_err_%": np.nanstd(errRel, axis=0),
        "mean_abs_err": np.nanmean(errAbs, axis=0),
    })
    summary
# endregion

## Scope / rejection report

In [ ]:
# region -> scope / rejection report (why each lane was dropped)
if runConfig["plot"]:
    rawObs = np.column_stack([
        (lastRow[:, sigIdx[o]] - obsOffset(o)) if o in sigIdx else np.full(Nrun, np.nan)
        for o in observations])
    rep = reporting.scopeRejectionReport(rawObs, paramMatrix, observations, plotParams, lim=divLim)
# endregion

## Per-observation error distribution

In [ ]:
# region -> boxplot: relative error distribution per observation across the converged sweep
if runConfig["plot"] and good.sum() > 0:
    figErr, ax = plt.subplots(figsize=(12, 5))
    ax.boxplot(errRel, tick_labels=utils.labelsFor(obsTargeted, "latex"), showfliers=runConfig["analysis"].get("showFliers", True))
    ax.axhline(0.0, color="k", lw=0.8)
    ax.axhline(pop["errorTarget"], color="r", ls="--", lw=0.8, label=f"±{pop['errorTarget']}% target")
    ax.axhline(-pop["errorTarget"], color="r", ls="--", lw=0.8)
    ax.set_ylabel("relative error (%)")
    ax.set_title(f"Convergence error across {good.sum()} valid lanes (all inductance levels)")
    ax.tick_params(axis="x", rotation=90)
    ax.legend()
    plt.tight_layout()
    plt.show()
# endregion


## Paper artifacts — error distribution figure + parameter-variation table


<details>
<summary>The per-observation error boxplot -> Images/figName, and the settled-parameter spread across the sweep -> generated/tableName (both guarded by paper.emit).</summary>

Both artifacts answer R1-C3 in the response letter. The figure is the error distribution
above: it shows the calibration reaching every target with an inductor in the model. The
table reads the run-end (settled) value of each calibrated parameter on every converged
lane and reports the value at the smallest and the largest inductance, the median, and the
span as a fraction of that median, so the reparametrisation induced by the inductor is
quantified parameter by parameter.
</details>


In [ ]:
# region -> paper artifacts: error-distribution figure + parameter range table vs the paper values
paper = runConfig["paper"]
if runConfig["plot"] and paper.get("emit", False) and good.sum() > 0:
    outFig = os.path.join(paper["imagesDir"], paper["figName"])
    outTab = os.path.join(paper["generatedDir"], paper["tableName"])

    figErr.savefig(outFig, dpi=200, bbox_inches="tight")
    print(f"figure -> {outFig}")

    # figure + caption as one \input-able block, so the document never carries a hand-written caption
    outFigTex = os.path.join(paper["generatedDir"], paper["figBlockName"])
    Lsym      = utils.labelFor(Lname, "latex", default=Lname)
    absErr    = np.abs(errRel)
    laneWorst = np.median(absErr.max(axis=1))
    jWorst    = int(absErr.max(axis=0).argmax())
    rcWorst   = (np.nanmax(np.abs((rcObs[0][targeted] - tgt) / tgt * 100.0))
                 if rcObs is not None else None)
    figCap = (
        f"Relative error of every calibration target at the end of calibration, over the "
        f"{int(good.sum())} calibrations of the aortic-inductance sweep ({Lsym} from "
        f"{Lvals[good].min():.4g} to {Lvals[good].max():.4g} mmHg\\,s$^2$/mL). Each box is the "
        f"interquartile range over the calibrations, the whiskers reach 1.5 times that range and "
        f"the calibrations beyond them are drawn individually. The dashed lines mark "
        f"$\\pm{pop['errorTarget']}\\%$. The largest error over the targets has a median of "
        f"{laneWorst:.3g}\\% across the sweep"
        + (f", against {rcWorst:.3g}\\% for the unmodified model" if rcWorst is not None else "")
        + f", and reaches {absErr[:, jWorst].max():.3g}\\% for "
          f"{utils.labelFor(obsTargeted[jWorst], 'latex', default=obsTargeted[jWorst])}.")
    figBlock = ("\\begin{center}\n"
                f"\\includegraphics[width=\\linewidth]{{{paper['figName']}}}\n\n"
                f"\\small\\textbf{{{paper.get('figLabel', 'Fig.~R2.')}}} {figCap}\n"
                "\\end{center}\n")
    with open(outFigTex, "w") as f:
        f.write(figBlock)
    print(f"figure block -> {outFigTex}")
    print(figBlock)

    # run-end parameter values on the converged lanes; rows keep the scenario (Fig. 6) order.
    # The paper value of each parameter is the RC anchor, i.e. the same calibration of the
    # unmodified model (it reproduces Table 5 of the paper to three digits).
    Lgood = Lvals[good]
    pGood = paramMatrix[good]
    def sig(x, d=3):
        """`d` significant figures, never in exponent form."""
        x = float(x)
        if x == 0.0:
            return "0"
        return f"{x:.{max(0, d - 1 - int(np.floor(np.log10(abs(x)))))}f}"

    boldPct = paper.get("boldPct", 10.0)      # rows moving more than this % of the paper value

    def bf(cell, on):
        """Bold a table cell; math needs \\boldmath to follow \\textbf."""
        if not on:
            return cell
        return f"{{\\boldmath\\textbf{{{cell}}}}}" if "$" in cell else f"\\textbf{{{cell}}}"

    rowsTex, rowsDF = {}, []
    for k, p in enumerate(plotParams):
        v      = pGood[:, k]
        lo, hi = v.min(), v.max()
        ref    = rcParams[p]
        pct    = (hi - lo) / abs(ref) * 100.0
        row    = [sig(ref), sig(lo), sig(hi), sig(hi - lo), sig(pct)]
        # parameter column uses the paper's own symbol (config/labels.json), not the code name
        rowsTex[bf(utils.labelFor(p, "latex", default=p), pct > boldPct)] = [
            bf(c, pct > boldPct) for c in row]
        rowsDF.append((p, *row, "bold" if pct > boldPct else ""))

    cols = ["Parameter", "Paper value", "Min", "Max", "Max $-$ Min",
            "Range \\%"]
    caption = (
        f"Range of each calibrated parameter over the aortic-inductance sweep "
        f"({int(good.sum())} converged lanes, {utils.labelFor(Lname, 'latex', default=Lname)} "
        f"from {Lgood.min():.4g} to "
        f"{Lgood.max():.4g} mmHg\\,s$^2$/mL), against the value the same calibration gives for "
        f"the unmodified model. Each value is read at the end of its calibration; the minimum "
        f"and the maximum are taken over the sweep and need not fall at its ends. The last "
        f"column, Range \\%, is the range as a percentage of the paper value, and the rows above "
        f"{sig(boldPct)}\\% of it are in bold.")

    # the response letter \inputs this file, so it is emitted as a non-float block carrying its
    # own manual label (paper.tableLabel), the style the letter uses for Fig. R1/R2
    label = paper.get("tableLabel", "Table~R1.")
    body  = utils.generate_latex_table_new(rowsTex, cols, "", "", "inductanceSweep", caption)
    body  = body.split("\\begin{tabular}", 1)[1].split("\\end{tabular}", 1)[0]
    table = ("\\begin{center}\n\\small\n\\begin{tabular}" + body + "\\end{tabular}\n\n"
             f"\\small\\textbf{{{label}}} {caption}\n\\end{{center}}\n")
    with open(outTab, "w") as f:
        f.write(table)
    print(f"table -> {outTab}")
    print(pd.DataFrame(rowsDF, columns=["param", "paper", "min", "max", "max-min",
                                       "%_of_paper", "emphasis"]).to_string(index=False))
    print(table)
# endregion


## Calibration convergence — observation traces vs target

<details>
<summary>Per-lane observation trajectory over the whole calibration vs its twin target (one line per L level).</summary>

</details>

In [ ]:
# region -> calibration convergence: per-lane observation traces vs target (one line per inductance level)
if runConfig["plot"] and good.sum() > 0:
    import h5py
    calib = modelStructure.get("calibration", {})
    paramForObs  = {calib[p]["params"]["varTarget"]: p for p in plotParams if p in calib}
    targetsByObs = {o: t for o, t in zip(observations, targetArr)}
    offsetsByObs = {o: off for o, off in zip(observations, offsetArr)}
    goodIdx = np.where(finiteRow)[0]
    targetPoints = runConfig["plotOpts"]["targetPoints"]

    # colorbar legend for the jet line coloring: line r <-> the L of the r-th plotted lane
    Lunit  = utils.labelFor(Lname, "unit", default="")
    Llabel = utils.labelFor(Lname, "plain", default=Lname) + (f" ({Lunit})" if Lunit else "")

    goodTraces, plotT, titleScope = {}, traceT, "converged window"
    if saveRaw:
        with h5py.File(outPath, "r") as f:
            if "raw_coarse" in f:                                   # fast path
                src, tkey, ds, titleScope = "raw_coarse", "raw_coarse_time", 1, "whole calibration"
            else:                                                   # slow fallback (full raw)
                src, tkey = "raw", "raw_time"
                ds, titleScope = max(1, f["raw"].shape[1] // targetPoints), "whole calibration (full raw)"
            sigN  = list(f["raw_signal_names"].asstr()[:])
            names = [o for o in obsTargeted if o in sigN]
            block = f[src][goodIdx, ::ds, :]
            plotT = f[tkey][::ds]
        goodTraces = {o: block[:, :, sigN.index(o)] for o in names}

    libPlots.plotCalibrationConvergence(
        goodTraces, traceT=plotT, targets=targetsByObs, offsets=offsetsByObs,
        paramForObs=paramForObs, runValues=Lvals[goodIdx], runValueLabel=Llabel,
        title=f"Calibration convergence ({titleScope}, {simulationParams['solver']['type']}) — per inductance level")
    plt.show()
# endregion

## Calibration convergence — calibrated parameters

<details>
<summary>Each calibrated parameter's trajectory over the whole calibration, one line per inductance level.</summary>

No target line (the params have no fixed target); this shows how the calibrator
re-solves each parameter as L changes.

</details>

In [ ]:
# region -> calibration convergence: one panel per calibrated parameter (one line per inductance level)
if runConfig["plot"] and good.sum() > 0:
    import h5py
    goodIdx = np.where(finiteRow)[0]
    targetPoints = runConfig["plotOpts"]["targetPoints"]

    Lunit  = utils.labelFor(Lname, "unit", default="")
    Llabel = utils.labelFor(Lname, "plain", default=Lname) + (f" ({Lunit})" if Lunit else "")

    paramTraces, plotT, titleScope = {}, traceT, "converged window"
    if saveRaw:
        with h5py.File(outPath, "r") as f:
            if "raw_coarse" in f:
                src, tkey, ds, titleScope = "raw_coarse", "raw_coarse_time", 1, "whole calibration"
            else:
                src, tkey = "raw", "raw_time"
                ds, titleScope = max(1, f["raw"].shape[1] // targetPoints), "whole calibration (full raw)"
            sigN  = list(f["raw_signal_names"].asstr()[:])
            names = [p for p in plotParams if p in sigN]
            block = f[src][goodIdx, ::ds, :]
            plotT = f[tkey][::ds]
        paramTraces = {p: block[:, :, sigN.index(p)] for p in names}

    libPlots.plotCalibrationConvergence(
        paramTraces, traceT=plotT, paramForObs=None, showLegend=False,
        runValues=Lvals[goodIdx], runValueLabel=Llabel,
        title=f"Calibration convergence -- parameters ({titleScope}, {simulationParams['solver']['type']}) — per inductance level")
    plt.show()
# endregion